# CPT Decoder — Google Colab (Stage 2 full run)

This notebook runs the full **Llama-3.2-3B + 4-bit QLoRA** training pipeline
on the 48,164-sentence LRS2 phoneme-to-text corpus via
`python -m src.cpt_decoder.dryrun`.

**Runtime estimate**
- Free Colab T4: ~3 hours for the 3-epoch full run (QLoRA resident VRAM ≈ 2 GB)
- Colab A100 (Pro): ~45 min

**Prerequisites**
- A Hugging Face token with read access to `meta-llama/Llama-3.2-3B`
  (gated; already approved for this project — generate one at
  https://huggingface.co/settings/tokens if you don't have one)
- ~12 GB free on Google Drive for checkpoints + metrics + sample generations

**Pipeline summary** (all paths relative to the repo root inside this notebook)
- `src/cpt_decoder/model.py` — auto-detects dtype (`fp16` on T4, `bf16` on Ampere+)
- `src/cpt_decoder/dryrun.py` — main training entry; reads every knob from `CPT_*` env vars
- `src/cpt_decoder/data/loader.py` — `_find_data_dir()` walks up looking for the
  marker CSV; works on Colab because the three CSVs are tracked in the repo
  under `src/cpt_decoder/data/`
- `RUNBOOK_real_run.md` — the uni-PC version of this run; env vars below are
  equivalent to Stage 2 §6 there

**Run order**: top-to-bottom, cell-by-cell. Cells 2-9 finish in ~10 minutes;
Cell 10 is the long training cell.

In [ ]:
# Mount Google Drive. The notebook writes checkpoints + metrics + sample
# generations under /content/drive/MyDrive/cpt_decoder/ so a disconnected
# runtime doesn't lose the trained adapter.
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# Install the CPT Decoder stack. peft + bitsandbytes + accelerate are the
# QLoRA path; nltk / jiwer / sacrebleu are reused from Phase 1 for evaluation;
# spacy + en_core_web_sm powers evaluation/contextual_analysis.py (Stage 3
# Option 3 — grammar-based error classification).
#
# Takes 2-3 min on a fresh Colab runtime. Colab's torch/transformers are
# pre-installed, so this only adds the extras listed in requirements-cpt-decoder.txt.
!pip install -q -r requirements-cpt-decoder.txt
!python -m spacy download en_core_web_sm 2>&1 | tail -3

In [ ]:
# Clone the lip_reading repo into /content and cd into it for every later cell.
# `%%bash` is used so the `cd` persists inside this cell and `os.chdir` is used
# in the Python process for subsequent cells in the same kernel.
%%bash
if [ ! -d "/content/lip_reading" ]; then
  git clone https://github.com/moe3n/lip_reading /content/lip_reading
else
  echo "Repo already cloned; pulling latest."
  cd /content/lip_reading && git pull
fi
echo "---"
cd /content/lip_reading && git log --oneline -3
echo "---"
cd /content/lip_reading && pwd

In [ ]:
# Make the Colab kernel's CWD point at /content/lip_reading so all later
# `python -m src.cpt_decoder.*` calls resolve correctly.
import os
os.chdir("/content/lip_reading")
print("CWD:", os.getcwd())

In [ ]:
# The three CSVs are tracked inside the repo (added in the 5 Jul 2026 pull)
# under src/cpt_decoder/data/. Verify them before training — without these
# data/loader.py raises FileNotFoundError at the first load_phoneme_text_pairs call.
!ls -la src/cpt_decoder/data/
!echo "---"
!du -sh src/cpt_decoder/data/*.csv

In [ ]:
# HF gated-repo login. This pops a Colab UI prompt — paste a READ token from
# https://huggingface.co/settings/tokens. The token never leaves this Colab
# runtime; it stays in ~/.cache/huggingface/token.
#
# If you prefer non-interactive auth, replace this cell with:
#   from huggingface_hub import login
#   login(token="hf_...")  # but DO NOT commit that token anywhere
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
# Confirm the GPU + the dtype auto-detect in src/cpt_decoder/model.py.
# Free Colab gives a T4 (compute capability 7.5 -> float16 path).
# Colab Pro A100 (CC 8.0 -> bfloat16 path).
# The auto-detect in _select_4bit_compute_dtype() handles both.
!nvidia-smi -L
!echo "---"
!python test_gpu.py

In [ ]:
# Re-run the 96/32 token-budget audit (verify_token_budget.py) against the
# full 48,164-row corpus with the real Llama 3.2 tokenizer. The fix to
# clean_phoneme_seq() (stripping "<space>" markers) brought the truncation
# rate at max_input_len=96 from 7.81% down to 0.00% — re-confirm it here
# before committing GPU time.
!python verify_token_budget.py 2>&1 | tail -20

In [ ]:
# Stage-2 env vars — equivalent to RUNBOOK_real_run.md §6.
# Every CPT_* knob below is read by src/cpt_decoder/dryrun.py in _env_str/int/float/bool.
import os

os.environ["CPT_MODEL_NAME"]        = "meta-llama/Llama-3.2-3B"
os.environ["CPT_N_HOMOPHONE"]       = "37374"
os.environ["CPT_N_NON_HOMOPHONE"]   = "10790"
os.environ["CPT_LORA_R"]            = "48"
os.environ["CPT_LORA_ALPHA"]        = "96"           # 2r convention from RUNBOOK §6
os.environ["CPT_LORA_DROPOUT"]      = "0.1"
os.environ["CPT_EPOCHS"]            = "3"
os.environ["CPT_BATCH_SIZE"]        = "2"
os.environ["CPT_GRAD_ACCUM"]        = "2"
os.environ["CPT_LEARNING_RATE"]     = "2e-4"
os.environ["CPT_WARMUP_STEPS"]      = "2"
os.environ["CPT_MAX_INPUT_LEN"]     = "96"
os.environ["CPT_MAX_TARGET_LEN"]    = "32"
os.environ["CPT_CONTRASTIVE_MARGIN"] = "0.5"
os.environ["CPT_CONTRASTIVE_LAMBDA"] = "0.1"
os.environ["CPT_CHECKPOINT_DIR"]    = "/content/drive/MyDrive/cpt_decoder/checkpoints"
os.environ["CPT_LLM_ERROR_JUDGE"]   = "1"            # opt-in Stage 3 Option 5

print("Stage-2 env vars set. CPT_CHECKPOINT_DIR =", os.environ["CPT_CHECKPOINT_DIR"])

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# LONG CELL — runs ~3 h on free T4, ~45 min on Colab Pro A100.
# ─────────────────────────────────────────────────────────────────────────
# What this does:
#   - Downloads meta-llama/Llama-3.2-3B safetensors (~6 GB, first run only)
#   - Quantises to NF4 4-bit at load time (residents ~1.5-2 GB VRAM)
#   - Injects LoRA r=48 adapters into q/k/v/o_proj
#   - Trains 3 epochs over 37374 homophone + 10790 non-homophone sentences
#     (80/20 train/val split = ~38,500 train rows, ~9,650 val rows)
#   - Saves adapter + tokenizer to CPT_CHECKPOINT_DIR (Drive)
#   - Generates on the full val split, computes WER/CER/BLEU4/EM
#   - Runs P2T error-pattern analysis (Stage 2 + Stage 3 Options 2/3/5)
#
# How to monitor progress while it runs:
#   - Each training step prints loss + s/step live (flush=True)
#   - A full copy of stdout is tee'd to /content/drive/MyDrive/cpt_decoder/training.log
#   - You can `tail -f` that file from another terminal / the Files pane
#
# If the runtime disconnects, just re-run this cell — CPT_CHECKPOINT_DIR is
# on Drive, so the resume is automatic.
!mkdir -p /content/drive/MyDrive/cpt_decoder/checkpoints
!python -m src.cpt_decoder.dryrun 2>&1 | tee /content/drive/MyDrive/cpt_decoder/training.log

In [ ]:
# Inspect what training wrote to Drive. The LoRA adapter is the small file
# (~10-25 MB) — base_model_name_or_path inside adapter_config.json should
# show meta-llama/Llama-3.2-3B, confirming the trained adapter is tied to the
# real gated model (not the Qwen stand-in).
!ls -lah /content/drive/MyDrive/cpt_decoder/checkpoints/
!echo "--- adapter_config.json ---"
!cat /content/drive/MyDrive/cpt_decoder/checkpoints/adapter_config.json

In [ ]:
# dryrun.py appends to metrics_log.csv after every run; mirror the latest
# copy into the Drive root for easy downloading.
import shutil, os
src = "/content/lip_reading/dryrun_checkpoints/metrics_log.csv"
# Some runs may write to the local default instead of Drive if CPT_CHECKPOINT_DIR
# wasn't picked up; handle both.
candidates = [
    "/content/drive/MyDrive/cpt_decoder/checkpoints/metrics_log.csv",
    src,
]
dst = "/content/drive/MyDrive/cpt_decoder/metrics_log.csv"
for c in candidates:
    if os.path.exists(c):
        shutil.copy(c, dst)
        print(f"Copied: {c} -> {dst}")
        break
else:
    print("metrics_log.csv not found in expected locations.")

print("--- last 10 rows ---")
!tail -10 /content/drive/MyDrive/cpt_decoder/metrics_log.csv 2>/dev/null || echo "(file missing)" 

In [ ]:
# Re-load the trained adapter + base model and run the P2T error-pattern
# analysis on a fresh validation pass. This calls evaluation/error_analysis.py
# (Stage 2 + Stage 3 Option 2/3) and evaluation/llm_judge.py (Stage 3 Option 5,
# already enabled via CPT_LLM_ERROR_JUDGE=1 in cell 9).
#
# Output:
#   - error_category_report() prints Homophone / Near-homophone / Other counts
#   - save_results() writes the metrics CSV
#   - Stage 3 Option 5 (LLM judge) re-classifies the substitutions the grammar
#     heuristic leaves unresolved
import os
os.chdir("/content/lip_reading")

from src.cpt_decoder import model as m
from src.cpt_decoder.data import loader as dl
from src.cpt_decoder.evaluation.error_analysis import error_category_report, print_error_report
from src.cpt_decoder.evaluation.metrics import stratified_evaluate, save_results

# Load the trained adapter back from Drive
CKPT = "/content/drive/MyDrive/cpt_decoder/checkpoints"
tokenizer = m.load_tokenizer(m.MODEL_NAME_TARGET)
model = m.load_model_with_lora(
    m.MODEL_NAME_TARGET, lora_r=int(os.environ.get("CPT_LORA_R", 48)),
    lora_alpha=int(os.environ.get("CPT_LORA_ALPHA", 96)),
    tokenizer=tokenizer,
)
# Load the LoRA weights from Drive on top of the base model
from peft import PeftModel
model = PeftModel.from_pretrained(model, CKPT, is_trainable=False)

# Build a small eval split (first 200 sentences — same shape as the dry run)
full_df = dl.load_original_phoneme_text_pairs()
homo_df, non_homo_df = dl.load_stratified_split(full_df)
df = __import__("pandas").concat([homo_df.head(100), non_homo_df.head(100)], ignore_index=True)
df = df.sample(frac=1, random_state=42).reset_index(drop=True)
homo_set = set(homo_df["sentence"])

refs, hyps, homo_mask = [], [], []
import torch
model.eval()
for _, row in df.iterrows():
    prompt = f"Phonemes: {row['phonemes']}\nText:"
    inputs = tokenizer(prompt, return_tensors="pt").to(m.DEVICE)
    with torch.no_grad():
        gen = model.generate(**inputs, max_new_tokens=24, do_sample=False,
                              pad_token_id=tokenizer.pad_token_id,
                              eos_token_id=tokenizer.eos_token_id,
                              repetition_penalty=1.3, no_repeat_ngram_size=3)
    decoded = tokenizer.decode(gen[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()
    refs.append(row["sentence"]); hyps.append(decoded)
    homo_mask.append(row["sentence"] in homo_set)

report = error_category_report(refs, hyps, homo_mask,
                                tokenizer=tokenizer, model=model, use_llm=True)
print_error_report(report, title="Stage 2 — error pattern analysis on re-eval pass")

In [ ]:
# Save 10 sample generations (input phonemes → model output → reference) to
# Drive as a CSV. Useful for the dissertation's qualitative-results appendix.
import os, csv
os.chdir("/content/lip_reading")
from src.cpt_decoder import model as m
from src.cpt_decoder.data import loader as dl
import torch

full_df = dl.load_original_phoneme_text_pairs()
homo_df, _ = dl.load_stratified_split(full_df)
sample = homo_df.head(10)

rows = []
model.eval()
for _, row in sample.iterrows():
    prompt = f"Phonemes: {row['phonemes']}\nText:"
    inputs = m.load_tokenizer(m.MODEL_NAME_TARGET)(prompt, return_tensors="pt").to(m.DEVICE)
    with torch.no_grad():
        gen = model.generate(**inputs, max_new_tokens=24, do_sample=False,
                              pad_token_id=tokenizer.pad_token_id,
                              eos_token_id=tokenizer.eos_token_id,
                              repetition_penalty=1.3, no_repeat_ngram_size=3)
    hyp = tokenizer.decode(gen[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()
    rows.append({"phonemes": row["phonemes"], "reference": row["sentence"], "generation": hyp})

out = "/content/drive/MyDrive/cpt_decoder/sample_generations.csv"
with open(out, "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=["phonemes", "reference", "generation"])
    w.writeheader()
    w.writerows(rows)
print(f"Wrote {len(rows)} sample generations to {out}")
print()
for r in rows:
    print(f"  PHON : {r['phonemes']}")
    print(f"  REF  : {r['reference']}")
    print(f"  GEN  : {r['generation']}")
    print()

## Next steps

**Done**
- Adapter checkpoint saved to `MyDrive/cpt_decoder/checkpoints/` (~10-25 MB)
- `metrics_log.csv` mirrored to `MyDrive/cpt_decoder/`
- `training.log` (full stdout) at `MyDrive/cpt_decoder/training.log`
- P2T error-pattern report printed
- 10 sample generations at `MyDrive/cpt_decoder/sample_generations.csv`

**Back on the uni PC** (re-use the trained adapter without re-training)
1. `git pull` (this notebook is the only tracked file change in this Colab
   session — nothing in `src/` was modified)
2. Copy the adapter folder from Drive into `lip_reading/dryrun_checkpoints/`
3. In a Python REPL:
   ```python
   from peft import PeftModel
   from transformers import AutoModelForCausalLM, AutoTokenizer
   base = AutoModelForCausalLM.from_pretrained("meta-llama/Llama-3.2-3B",
                  quantization_config=bnb_cfg, device_map="auto")
   tok = AutoTokenizer.from_pretrained("dryrun_checkpoints")
   model = PeftModel.from_pretrained(base, "dryrun_checkpoints", is_trainable=False)
   ```

**Re-running on Colab with different settings**
- Edit the env vars in cell 9 (e.g. `CPT_EPOCHS=5`, `CPT_LORA_R=64`) and re-run
  from cell 10 onward. The existing checkpoint is overwritten on each run.

**Refreshing CSVs from upstream**
- `git pull` in cell 4 pulls the latest tracked CSVs from the repo.
  If you maintain a separate upstream of LRS2 data, replace the files in
  `src/cpt_decoder/data/` directly and re-run from cell 5.

**Cleanup**
- The HF token lives in `~/.cache/huggingface/token` inside the Colab runtime.
  It is destroyed when the runtime disconnects. No token is written to Drive
  by this notebook.